In [34]:
import numpy as np
import math
import pandas as pd


board = np.zeros((3,3), dtype='int')

print(board)

In [35]:
board = np.zeros([3,3], dtype = int)
    
print(board)

[[0 0 0]
 [0 0 0]
 [0 0 0]]


In [36]:

isPlayDone = False
def reset(board):
    board[:] = 0
    
def hasWonRewardFunc(board):
    # row
    for i in board:
        if(i[0] != 0 and np.all(i == i[0])):
            return i[0]

    # column
    for i in board.T:
        if(i[0] != 0 and np.all(i == i[0])):
            return i[0]

    # diagonal 1
    if(board[0][0] != 0 and board[0][0] == board[1][1] and board[0][0] == board[2][2]):
        return board[0][0]

    # diagonal 2
    if(board[0][2] != 0 and board[0][2] == board[1][1] and board[0][2] == board[2][0]):
        return board[0][2]

    return 0

def step(board, player, i,j):
    board[i][j] = player
    
def strategyOppositePlayer(board):

    row_to_block = (
    (np.count_nonzero(board == 1, axis=1) == 2) &
    (np.count_nonzero(board == 0, axis=1) == 1))

    rows = np.where(row_to_block)[0]

    if len(rows) > 0:
        r = rows[0]
        c = np.where(board[r] == 0)[0][0]
        return r,c

    column_to_block = (
    (np.count_nonzero(board == 1, axis=0) == 2) &
    (np.count_nonzero(board == 0, axis=0) == 1))
    
    cols = np.where(column_to_block)[0]
    if len(cols) > 0:
        c = cols[0]
        r = np.where(board[:, c] == 0)[0][0]
        return r, c

    #Block diagonal
    diag = np.array([board[0,0], board[1,1], board[2,2]])
    if np.count_nonzero(diag == 1) == 2 and np.count_nonzero(diag == 0) == 1:
        i = np.where(diag == 0)[0][0]
        return [(0,0), (1,1), (2,2)][i]
    
    anti = np.array([board[0,2], board[1,1], board[2,0]])
    if np.count_nonzero(anti == 1) == 2 and np.count_nonzero(anti == 0) == 1:
        i = np.where(anti == 0)[0][0]
        return [(0,2), (1,1), (2,0)][i]
        
    #check the possibility of straight vertical lines
    for idx, i in enumerate(board):
        if(np.any(i == -1) and not np.any(i == 1)):
            for idy, j in enumerate(i):
                if(j == 0):
                    return idx, idy
    for idx, i in enumerate(board.T):
        if(np.any(i == -1) and not np.any(i == 1)):
            for idy, j in enumerate(i):
                if(j == 0):
                    return idy, idx
    
    #Check if diagonal is formed
    if((board[0][0] == -1 or board[1][1] == -1 or board[2][2] == -1) and (board[0][0] != 1 and board[1][1] != 1 and board[2][2] != 1)):
        if(board[0][0] == 0):
            return 0,0;
        if(board[1][1] == 0):
            return 1,1;  
        if(board[2][2] == 0):
            return 2,2;
    #Check if vertical diagonal is formed
    if((board[0][2] == -1 or board[1][1] == -1 or board[2][0] == -1) and (board[0][2] != 1 and board[1][1] != 1 and board[2][0] != 1)):
        if(board[0][2] == 0):
            return 0,2;
        if(board[1][1] == 0):
            return 1,1;  
        if(board[2][0] == 0):
            return 2,0;
    # return position i, j\
    empty = np.argwhere(board == 0)

    if len(empty) > 0:
        return tuple(empty[np.random.randint(len(empty))])


def exploringOppositePlayer(board, epsilon=0.3):
    """
    An opponent that balances playing smart with pure exploration.
    epsilon: The probability (0.0 to 1.0) of playing a completely random move.
    """
    empty_spots = np.argwhere(board == 0)
    
    # Base case: Board is full
    if len(empty_spots) == 0:
        return None

    # ==========================================
    # 1. EXPLORATION PHASE
    # ==========================================
    # With probability 'epsilon', ignore strategy and pick a random valid move
    if np.random.uniform(0, 1) < epsilon:
        random_choice = empty_spots[np.random.choice(len(empty_spots))]
        return tuple(random_choice)

    # ==========================================
    # 2. EXPLOITATION PHASE (Play Smart)
    # ==========================================
    # Helper function: Finds a winning or blocking move using array summation
    # If target is -1 (Opponent), it looks for a win.
    # If target is 1 (Bot), it looks for a block.
    def find_threat(target_player):
        # Rows
        for i in range(3):
            if np.sum(board[i, :]) == target_player * 2 and np.count_nonzero(board[i, :] == 0) == 1:
                return i, np.where(board[i, :] == 0)[0][0]
        # Columns
        for j in range(3):
            if np.sum(board[:, j]) == target_player * 2 and np.count_nonzero(board[:, j] == 0) == 1:
                return np.where(board[:, j] == 0)[0][0], j
        # Main Diagonal
        diag = np.diag(board)
        if np.sum(diag) == target_player * 2 and np.count_nonzero(diag == 0) == 1:
            idx = np.where(diag == 0)[0][0]
            return idx, idx
        # Anti-Diagonal
        anti_diag = np.diag(np.fliplr(board))
        if np.sum(anti_diag) == target_player * 2 and np.count_nonzero(anti_diag == 0) == 1:
            idx = np.where(anti_diag == 0)[0][0]
            return idx, 2 - idx
            
        return None

    # Priority A: Take the win if available
    win_move = find_threat(-1)
    if win_move:
        return win_move

    # Priority B: Block the bot if the bot is about to win
    block_move = find_threat(1)
    if block_move:
        return block_move

    # Priority C: Take the center if it is open (Standard heuristic)
    if board[1][1] == 0:
        return 1, 1

    # Priority D: If no obvious threats, pick a random open spot
    random_choice = empty_spots[np.random.choice(len(empty_spots))]
    return tuple(random_choice)


def calculateCurrentBoardConfig(board):
    num = 0
    for i in range(3):
        for j in range(3):
            pos = 3*i + j

            if board[i][j] == -1:
                digit = 1
            elif board[i][j] == 1:
                digit = 2
            else:
                digit = 0

            num += digit * (3 ** pos)

    return int(num)
                
def isValidStep(board, i, j):
    return board[i][j] == 0

def findMove(i):
    return divmod(i, 3)

In [53]:

## Learning with Epsilon explorating opposing player

reset(board)
initLearning = np.zeros([19683,9], dtype = float)
learnRate = 0.25
discountFactor = 0.05

def learnWithExploration(myBoard,previousStep,previousIndex, n):
    if(n > 9):
        return
    move = exploringOppositePlayer(myBoard)
    if move is None:
        reset(myBoard)
        return
    i, j = move
    currentIndex = calculateCurrentBoardConfig(myBoard)    
    step(myBoard, -1, i, j) # play strategy player
    whoWon = hasWonRewardFunc(myBoard)
    if whoWon == -1:
        # -ve reward
        initLearning[previousIndex][previousStep] = initLearning[previousIndex][previousStep] + learnRate* (-1 - initLearning[previousIndex][previousStep])
        return
    newBoardIndex = calculateCurrentBoardConfig(myBoard)

    # dont wait till the end to update the states of the game

    if previousStep is not None:
        initLearning[previousIndex][previousStep] = initLearning[previousIndex][previousStep] + learnRate* (discountFactor *np.max(initLearning[newBoardIndex]) - initLearning[previousIndex][previousStep])
    
    possibleMoves = np.argsort(initLearning[newBoardIndex])[::-1]
    k,l = 0,0
    for possibleMove in possibleMoves:
        k,l = findMove(possibleMove)
        if not isValidStep(myBoard, k, l): #Not a valid move
            continue
        currentIndex = calculateCurrentBoardConfig(myBoard)    
        temp_board = myBoard.copy() 
        step(temp_board, 1, k, l) # play bot
        whoWon = hasWonRewardFunc(temp_board)
        if whoWon==0:
            if np.all(temp_board != 0):
                initLearning[currentIndex][possibleMove] = initLearning[currentIndex][possibleMove] + learnRate* (-initLearning[currentIndex][possibleMove])
                continue
        if whoWon == 1:
            # +ve reward
            initLearning[currentIndex][possibleMove] = initLearning[currentIndex][possibleMove] + learnRate* (1 -initLearning[currentIndex][possibleMove])
            return
        
        learnWithExploration(temp_board, possibleMove,currentIndex, n+1)# given step is taken, learn all possible combinations

for i in range(1000):
    reset(board)
    learnWithExploration(board, None, None, 0)
df_describe = pd.DataFrame(initLearning)
df_describe.describe()
 

,0,1,2,3,4,5,6,7,8
count,19683.000000,19683.000000,19683.000000,19683.000000,19683.000000,19683.000000,19683.000000,19683.000000,19683.000000
mean,-0.007216,-0.009699,-0.007357,-0.009892,-0.002596,-0.010224,-0.008034,-0.010368,-0.007350
std,0.107889,0.101802,0.107994,0.103644,0.058944,0.104317,0.110668,0.103657,0.109585
min,-1.000000,-1.000000,-1.000000,-1.000000,-0.957765,-1.000000,-1.000000,-1.000000,-1.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,0.924915,1.000000,1.000000,1.000000,1.000000


In [51]:
#Learning with Self Play
oppositionLearning = np.zeros([19683,9], dtype = float)

newSelfPlayLearning = np.zeros([19683,9], dtype = float)
reset(board)

def selfPlayLearning(current_board):
    # 1. Convert the current board to its index
    state_index = calculateCurrentBoardConfig(current_board)
    
    # 2. Get the 9 Q-values for this specific state
    q_values = oppositionLearning[state_index]
    
    # 3. Sort the moves from highest expected value to lowest
    best_moves = np.argsort(q_values)[::-1]
    
    # 4. Iterate through the best moves and pick the first one that is valid
    for move in best_moves:
        i, j = findMove(move)
        if isValidStep(current_board, i, j):
            return i, j
            
    return None # Fallback if no moves are valid
        

def learnWithSelfPlay(myBoard,previousStep,previousIndex, n):
    if(n > 9):
        return
    move = selfPlayLearning(myBoard)
    if move is None:
        reset(myBoard)
        return
    i, j = move
    currentIndex = calculateCurrentBoardConfig(myBoard)    
    step(myBoard, -1, i, j) # play strategy player
    whoWon = hasWonRewardFunc(myBoard)
    if whoWon == -1:
        # -ve reward
        newSelfPlayLearning[previousIndex][previousStep] = newSelfPlayLearning[previousIndex][previousStep] + learnRate* (-1 - newSelfPlayLearning[previousIndex][previousStep])
        return
    newBoardIndex = calculateCurrentBoardConfig(myBoard)

    # dont wait till the end to update the states of the game

    if previousStep is not None:
        newSelfPlayLearning[previousIndex][previousStep] = newSelfPlayLearning[previousIndex][previousStep] + learnRate* (discountFactor *np.max(newSelfPlayLearning[newBoardIndex]) - newSelfPlayLearning[previousIndex][previousStep])
    
    possibleMoves = np.argsort(newSelfPlayLearning[newBoardIndex])[::-1]
    k,l = 0,0
    for possibleMove in possibleMoves:
        k,l = findMove(possibleMove)
        if not isValidStep(myBoard, k, l): #Not a valid move
            continue
        currentIndex = calculateCurrentBoardConfig(myBoard)    
        temp_board = myBoard.copy() 
        step(temp_board, 1, k, l) # play bot
        whoWon = hasWonRewardFunc(temp_board)
        if whoWon==0:
            if np.all(temp_board != 0):
                newSelfPlayLearning[currentIndex][possibleMove] = newSelfPlayLearning[currentIndex][possibleMove] + learnRate* (-newSelfPlayLearning[currentIndex][possibleMove])
                continue
        if whoWon == 1:
            # +ve reward
            newSelfPlayLearning[currentIndex][possibleMove] = newSelfPlayLearning[currentIndex][possibleMove] + learnRate* (1 -newSelfPlayLearning[currentIndex][possibleMove])
            return
        
        learnWithExploration(temp_board, possibleMove,currentIndex, n+1)# given step is taken, learn all possible combinations


for i in range(1000):
    reset(board)
    oppositionLearning=newSelfPlayLearning.copy()
    learnWithSelfPlay(board, None, None, 0)
    
df_describe = pd.DataFrame(initLearning)
df_describe.describe()

,0,1,2,3,4,5,6,7,8
count,19683.000000,19683.000000,19683.000000,19683.000000,19683.000000,19683.000000,19683.000000,19683.000000,19683.000000
mean,-0.008759,-0.011063,-0.008362,-0.011401,-0.003682,-0.011485,-0.008741,-0.011462,-0.008336
std,0.117856,0.115296,0.121593,0.117701,0.104184,0.113629,0.124826,0.115107,0.112372
min,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [54]:
# Helper to get the best bot move from your trained table
def get_bot_move(current_board):
    state_index = calculateCurrentBoardConfig(current_board)
    q_values = initLearning[state_index]
    #q_values = newSelfPlayLearning[state_index]
    
    best_moves = np.argsort(q_values)[::-1]
    
    for move in best_moves:
        i, j = findMove(move)
        if isValidStep(current_board, i, j):
            return i, j
            
def print_board(board):
    # Dictionary to map numbers to symbols
    symbols = {0: '.', 1: 'X', -1: 'O'}
    print("\n  0 1 2 (Columns)")
    for i in range(3):
        row_str = " ".join([symbols[board[i][j]] for j in range(3)])
        print(f"{i} {row_str}")
    print()

def play_text_game():
    current_board = np.zeros((3, 3), dtype=int)
    print("Game Start! You are 'O'. Bot is 'X'.")
    print("Enter your moves as 'row col' (e.g., '1 1' for the center).")
    
    while True:
        print_board(current_board)
        
        # --- Human Turn ---
        user_input = input("Your move (row col): ")
        
        try:
            r, c = map(int, user_input.split())
            if r < 0 or r > 2 or c < 0 or c > 2 or not isValidStep(current_board, r, c):
                print("Invalid move! That space is taken or out of bounds. Try again.")
                continue
        except ValueError:
            print("Invalid format! Please enter two numbers separated by a space (e.g., 0 2).")
            continue
            
        step(current_board, -1, r, c)
        
        # Check if Human won (unlikely against your bot!) or Draw
        if hasWonRewardFunc(current_board) == -1:
            print_board(current_board)
            print("🎉 You won!")
            break
        elif np.all(current_board != 0):
            print_board(current_board)
            print("🤝 It's a Draw!")
            break
            
        # --- Bot Turn ---
        print("\nBot is thinking...")
        bot_move = get_bot_move(current_board)
        
        if bot_move:
            step(current_board, 1, bot_move[0], bot_move[1])
            print(f"Bot places 'X' at {bot_move[0]} {bot_move[1]}")
            
        # Check if Bot won
        if hasWonRewardFunc(current_board) == 1:
            print_board(current_board)
            print("🤖 Bot wins!")
            break
        elif np.all(current_board != 0):
            print_board(current_board)
            print("🤝 It's a Draw!")
            break

# Start the game
play_text_game()

Game Start! You are 'O'. Bot is 'X'.
Enter your moves as 'row col' (e.g., '1 1' for the center).

  0 1 2 (Columns)
0 . . .
1 . . .
2 . . .



Your move (row col):  1 1



Bot is thinking...
Bot places 'X' at 1 2

  0 1 2 (Columns)
0 . . .
1 . O X
2 . . .



Your move (row col):  0 2



Bot is thinking...
Bot places 'X' at 2 0

  0 1 2 (Columns)
0 . . O
1 . O X
2 X . .



...

Your move (row col):  0 1


Invalid move! That space is taken or out of bounds. Try again.

  0 1 2 (Columns)
0 X X O
1 . O X
2 X O O



Your move (row col):  0 2


Invalid move! That space is taken or out of bounds. Try again.

  0 1 2 (Columns)
0 X X O
1 . O X
2 X O O



KeyboardInterrupt: Interrupted by user